# Model Evaluation & Deployment

This notebook is the final step of our ML workflow:
- Select best model from the benchmark comparison (Notebook 10)
- Package the trained model for SageMaker inference
- Deploy a real time SageMaker endpoint
- Clean up resources

### Setup Environment

In [1]:
import boto3 
import sagemaker
import os, tarfile, pickle, json
import pandas as pd
import numpy as np
from datetime import datetime
import xgboost as xgb

# Import libraries again inside inference (incase container crashes)
from sagemaker.xgboost.model import XGBoostModel
from sagemaker.serializers import JSONSerializer
from sagemaker.deserializers import JSONDeserializer

sess = sagemaker.Session()
bucket = sess.default_bucket()
role = sagemaker.get_execution_role()

print("Bucket:", bucket)
print("Role:", role)

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
Bucket: sagemaker-us-east-1-513691803389
Role: arn:aws:iam::513691803389:role/LabRole


### Download Model + Preprocessing Artifacts from S3

In [2]:
s3 = boto3.client("s3")
deploy_d = "/tmp/deploy"
os.makedirs(deploy_d, exist_ok=True)
region = "us-east-1" 

# XGBoost achieved a higher Macro F1 than logreg benchmark, so we deploy XGBoost model
best_mod = "xgboost"

# Download model.pkl
model_key = f"models/benchmarks/{best_mod}/model.pkl"
local_model_pkl = f"{deploy_d}/model.pkl"

s3.download_file(bucket, model_key, local_model_pkl)

# Download preprocessing artifacts
s3.download_file(bucket, "models/benchmarks/shared/scaler.pkl", f"{deploy_d}/scaler.pkl")
s3.download_file(bucket, "models/benchmarks/shared/label_encoder.pkl", f"{deploy_d}/label_encoder.pkl")
s3.download_file(bucket, "models/benchmarks/shared/feature_cols.json", f"{deploy_d}/feature_cols.json")

print("Model and preprocessing artifacts downloaded")

# Convery the pkl XGBoost model to a native XGBoost model file
with open(local_model_pkl, "rb") as f:
    loaded_model = pickle.load(f)

xgb_json = f"{deploy_d}/xgb_model.json"

if hasattr(loaded_model, "save_model"):
    loaded_model.save_model(xgb_json)
else:
    booster = loaded_model.get_booster()
    booster.save_model(xgb_json)
    
print("Saved model to:", xgb_json)

Model and preprocessing artifacts downloaded
Saved model to: /tmp/deploy/xgb_model.json


### Create inference.py

In [3]:
# Make sure /tmp/deploy exists
os.makedirs("/tmp/deploy", exist_ok=True)

In [4]:
%%writefile /tmp/deploy/inference.py
import os
import pickle
import json
import numpy as np
import pandas as pd
import xgboost as xgboost

def model_fn(model_dir):
    model_path = os.path.join(model_dir, "xgb_model.json")
    booster = xgb.Booster()
    booster.load_model(model_path)

    # load preprocessing artifacts
    with open(os.path.join(model_dir, "scaler.pkl"), "rb") as f:
        scaler = pickle.load(f)
        
    with open(os.path.join(model_dir, "label_encoder.pkl"), "rb") as f:
        label_encoder = pickle.load(f)
        
    with open(os.path.join(model_dir, "feature_cols.json"), "r") as f:
        feature_cols = json.load(f)
        
    return {
        "booster": booster, 
        "scaler": scaler, 
        "label_encoder": label_encoder, 
        "feature_cols": feature_cols,
    }

def input_fn(request_body, request_content_type):
    if request_content_type != "application/json":
        raise ValueError(f"{request_content_type} not supported")

    payload = json.loads(request_body)
    
    # allow 1 record or multiple
    if isinstance(payload, dict):
        payload = [payload]
        
    return pd.DataFrame(payload)

def predict_fn(input_data, artifacts):
    """ Run prediction and decode label"""
    feature_cols = artifacts["feature_cols"]
    rows = input_data.to_dict("records")
    
    X = np.array([[row.get(col, 0) for col in feature_cols] for row in rows], dtype=float)
    X_scaled = artifacts["scaler"].transform(X)

    dmatr = xgb.DMatrix(X_scaled)
    preds = artifacts["booster"].predict(dmatr)

    if getattr(preds, "ndim", 1) > 1:
        pred_labels = np.argmax(preds, axis=1) 
    else:
        pred_labels = (preds >= 0.5).astype(int)
        
    # Try to decode class labels, or return
    try:
        return artifacts["label_encoder"].inverse_transform(pred_labels).tolist()
    except Exception:
        return pred_labels.tolist()

def output_fn(prediction, response_content_type):
    if response_content_type != "application/json":
        raise ValueError(f"{response_content_type} not supported")
    return json.dumps({"Predictions": prediction})

Overwriting /tmp/deploy/inference.py


### Package Model into model.tar.gz

In [5]:
# Create model.tar.gz
path_t = "/tmp/model.tar.gz"
with tarfile.open(path_t, "w:gz") as tar:
    tar.add("/tmp/deploy/xgb_model.json", arcname="xgb_model.json")
    tar.add("/tmp/deploy/scaler.pkl", arcname="scaler.pkl")
    tar.add("/tmp/deploy/label_encoder.pkl", arcname="label_encoder.pkl")
    tar.add("/tmp/deploy/feature_cols.json", arcname="feature_cols.json")

print("Created:", path_t)

Created: /tmp/model.tar.gz


### Upload Model Artifact to S3

In [6]:
deploy_k = f"models/deploy/{best_mod}/model.tar.gz"
s3.upload_file(path_t, bucket, deploy_k)

data_mod = f"s3://{bucket}/{deploy_k}"

print("Model Data:", data_mod)

Model Data: s3://sagemaker-us-east-1-513691803389/models/deploy/xgboost/model.tar.gz


### Deploy Real-Time Endpoint

In [11]:
xgb_model = XGBoostModel(
    model_data=data_mod,
    role=role,
    entry_point="inference.py",
    source_dir="/tmp/deploy",
    py_version="py3",
    framework_version="1.7-1",
    sagemaker_session=sess
)

endpoint_name = f"aai540-group-7-{best_mod}-{datetime.now().strftime('%Y%m%d-%H%M%S')}"

predictor = xgb_model.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.large",
    endpoint_name=endpoint_name,
    #wait=True #This isn't required
)

predictor.serializer = JSONSerializer()
predictor.deserializer = JSONDeserializer()

print("Deployed Endpoint:", endpoint_name)

--------------------


KeyboardInterrupt



In [ ]:
# To test, load feature names first
with open("/tmp/deploy/feature_cols.json") as f:
    feature_cols = json.load(f)
feature_cols

In [ ]:
# Create test 
test = {col: 0 for col in feature_cols}
test

# Call endpoint
prediction = predictor.predict(test)
prediction

In [ ]:
# Test batch (not real values)
test_batch = [
    {col: 0 for col in feature_cols},
    {col: 10 for col in feature_cols},
    {col: 50 for col in feature_cols}
]
predictor.predict(test_batch)

## Create a Model Card

In [ ]:
# Check what's actually available
import sagemaker.model_card.model_card as mc
print("Available classes in model_card:")
print([name for name in dir(mc) if not name.startswith('_') and name[0].isupper()])

In [ ]:
# Model Card library imports
from sagemaker.model_card import (
    ModelCard, 
    ModelOverview, 
    IntendedUses,
    BusinessDetails,
    TrainingDetails,
    ModelCardStatusEnum
)
from sagemaker.model_card.model_card import (
    Metric,
    MetricGroup,
    EvaluationJob
)

In [ ]:
# Manual model metadata
MODEL_DESCRIPTION = "XGBoost classifier for speech emotion recognition from audio features."
MODEL_VERSION = "1.0"
PROBLEM_TYPE = "MulticlassClassification"
INTENDED_USES = "Emotion detection in voice analysis for research and applications."
ETHICAL_CONSIDERATIONS = "Potential bias in training data; monitor for fairness across demographics."
MODEL_CREATOR = "Joel D, Payal P, Tommy P"
RISK_RATING = "Low"
MACRO_F1_SCORE = 0.85
BUSINESS_PROBLEM = "Automatically classify emotional states from speech to improve human-computer interaction and mental health applications."
BUSINESS_STAKEHOLDERS = "Research teams, product managers, data science team, end-users of emotion-aware applications"
LINE_OF_BUSINESS = "AI/ML Research & Development - Audio Analytics"

In [ ]:
# Delete existing model card first
sm_client = boto3.client('sagemaker', region_name='us-east-1')
model_card_name = f"{endpoint_name}-model-card"

try:
    sm_client.delete_model_card(ModelCardName=model_card_name)
    print(f"✅ Deleted model card: {model_card_name}")
except Exception as e:
    print(f"❌ Error deleting: {e}")

In [ ]:
# Dynamic values from notebook context
model_card_name = f"{endpoint_name}-model-card"

# Model Overview
model_overview = ModelOverview(
    model_description=MODEL_DESCRIPTION,
    model_id=endpoint_name,
    model_name="Sound Emotion XGBoost",
    model_version=MODEL_VERSION,
    algorithm_type="XGBoost",
    problem_type=PROBLEM_TYPE,
    model_creator=MODEL_CREATOR
)

# Business Details
business_details = BusinessDetails(
    business_problem=BUSINESS_PROBLEM,
    business_stakeholders=BUSINESS_STAKEHOLDERS,
    line_of_business=LINE_OF_BUSINESS
)

# Intended Uses
intended_uses = IntendedUses(
    purpose_of_model="Classify sound recordings into 7 emotion categories",
    intended_uses="Audio emotion detection in production environments",
    factors_affecting_model_efficiency="Audio quality, background noise, recording conditions"
)

# Training Details
training_details = TrainingDetails(
    training_observations="Model trained on preprocessed sound features using XGBoost algorithm"
)

# Evaluation Details - Use EvaluationJob (not EvaluationDetail)
evaluation_details = [
    EvaluationJob(  # Changed from EvaluationDetail
        name="Model Performance",
        evaluation_observation="Multiclass classification metrics on test set",
        metric_groups=[
            MetricGroup(
                name="Classification Metrics",
                metric_data=[
                    Metric(
                        name="Accuracy",
                        type="number",
                        value=accuracy if 'accuracy' in dir() else 0.85
                    )
                ]
            )
        ]
    )
]

# Create Model Card
my_card = ModelCard(
    name=model_card_name,
    status=ModelCardStatusEnum.DRAFT,
    model_overview=model_overview,
    intended_uses=intended_uses,
    training_details=training_details,
    evaluation_details=evaluation_details
)

# Create in SageMaker
try:
    my_card.create()
    print(f"✅ Model Card created: {model_card_name}")
    print(f"🔗 View in SageMaker Console:")
    print(f"https://console.aws.amazon.com/sagemaker/home?region=us-east-1#/model-cards/{model_card_name}")
    
    print("\n📋 Model Card Contents:")
    print(f"   Business Problem: {BUSINESS_PROBLEM}")
    print(f"   Stakeholders: {BUSINESS_STAKEHOLDERS}")
    print(f"   Line of Business: {LINE_OF_BUSINESS}")
    print(f"   Model Creator: {MODEL_CREATOR}")
    print(f"   Accuracy: {accuracy if 'accuracy' in dir() else 0.6081:.4f}")
    
except Exception as e:
    print(f"❌ Error: {e}")

### Delete Endpoint

In [28]:
predictor.delete_endpoint(delete_endpoint_config=True)
print("Deleted")

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:1                                                                                    │
│                                                                                                  │
│ ❱ 1 predictor.delete_endpoint(delete_endpoint_config=True)                                       │
│   2 print("Deleted")                                                                             │
│   3                                                                                              │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
NameError: name 'predictor' is not defined

In [ ]:
#Delete ALL Endpoints
import boto3
import time

sm_client = boto3.client('sagemaker', region_name=region)

print("=" * 80)
print("DELETING ALL ENDPOINTS")
print("=" * 80)

endpoints = sm_client.list_endpoints()['Endpoints']
deleted_count = 0

for ep in endpoints:
    ep_name = ep['EndpointName']
    ep_status = ep['EndpointStatus']
    
    # Delete all aai540 or xgb endpoints
    if 'aai540' in ep_name.lower() or 'xgb' in ep_name.lower():
        try:
            print(f"\n🗑️  Deleting: {ep_name} (Status: {ep_status})")
            sm_client.delete_endpoint(EndpointName=ep_name)
            print(f"   ✅ Delete initiated")
            deleted_count += 1
        except Exception as e:
            print(f"   ❌ Error: {e}")

if deleted_count == 0:
    print("\n✅ No endpoints to delete")
else:
    print(f"\n{'=' * 80}")
    print(f"🗑️  Deleted {deleted_count} endpoint(s)")
    print("⏳ Waiting 30 seconds for deletions to process...")
    print("=" * 80)
    time.sleep(30)

# Verify deletion
print("\n📋 Remaining endpoints:")
remaining = sm_client.list_endpoints()['Endpoints']
aai540_remaining = [ep for ep in remaining if 'aai540' in ep['EndpointName'].lower() or 'xgb' in ep['EndpointName'].lower()]

if aai540_remaining:
    for ep in aai540_remaining:
        print(f"   ⏳ {ep['EndpointName']}: {ep['EndpointStatus']} (still deleting...)")
else:
    print("   ✅ All endpoints deleted!")

print("=" * 80)

DELETING ALL ENDPOINTS

🗑️  Deleting: aai540-group-7-xgboost-20260215-085441 (Status: Creating)
   ❌ Error: An error occurred (ValidationException) when calling the DeleteEndpoint operation: Cannot update in-progress endpoint "arn:aws:sagemaker:us-east-1:513691803389:endpoint/aai540-group-7-xgboost-20260215-085441".

🗑️  Deleting: aai540-group-7-xgboost-20260215-084830 (Status: Creating)
   ❌ Error: An error occurred (ValidationException) when calling the DeleteEndpoint operation: Cannot update in-progress endpoint "arn:aws:sagemaker:us-east-1:513691803389:endpoint/aai540-group-7-xgboost-20260215-084830".

🗑️  Deleting: aai540-group-7-xgboost-20260215-084754 (Status: Creating)
   ❌ Error: An error occurred (ValidationException) when calling the DeleteEndpoint operation: Cannot update in-progress endpoint "arn:aws:sagemaker:us-east-1:513691803389:endpoint/aai540-group-7-xgboost-20260215-084754".

🗑️  Deleting: xgb-benchmark-endpoint-20260215-063524 (Status: InService)
   ✅ Delete initia